# Run SparseGPT, GPTQ, and SmoothQuant crosscoder experiments

This notebook creates the new compressed checkpoints and runs the existing crosscoder pipeline while preserving the repository's result naming and skip/resume behavior.

New methods:
- `sparsegpt`: 50% unstructured pruning with Hessian-guided error compensation.
- `gptq`: 4-bit grouped GPTQ, stored as fake-quantized weights for accuracy experiments.
- `smoothquant`: W8A8 fake quantization with runtime activation quantization (not accelerated INT8 inference).

The expensive cells are disabled by default. Review the selected models, methods, and components, then set the relevant `RUN_*` flag to `True`.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

# Locate the repository whether Jupyter starts in the repo root or notebooks/.
REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / "src").exists(), "Start this notebook from the repository or notebooks directory."

SRC_DIR = REPO_ROOT / "src"
for path in (REPO_ROOT, SRC_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
os.chdir(REPO_ROOT)

print(f"Repository: {REPO_ROOT}")
print(f"Python: {sys.executable}")

## 1. Prerequisites

Install the repository dependencies and prepare Visual-Counterfact before running compression:

```bash
pip install -r requirements.txt
python preprocessing/setup_crosscoder_from_hf.py --dataset-only
```

The calibration dataset must exist at `output/counterfactual_selected/`. Compression is GPU-intensive; BLIP V-only is the smallest useful smoke test.

In [ ]:
import torch

required_data = REPO_ROOT / "output" / "counterfactual_selected"
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Calibration data exists: {required_data.exists()} ({required_data})")

if not required_data.exists():
    print("Run: python preprocessing/setup_crosscoder_from_hf.py --dataset-only")

## 2. Select the experiment matrix

Compression labels use `V+P`, while crosscoder result directories and CLI use `V_P`. The mapping below handles that difference. Start with `blip2`, `sparsegpt`, and `V` for a smoke run, then expand to the full lists.

In [ ]:
NEW_METHODS = ["sparsegpt", "gptq", "smoothquant"]
ALL_MODELS = ["blip2", "qwen3vl", "llava15"]
ALL_COMPRESSION_COMPONENTS = ["V", "P", "V+P"]

# Safe smoke-test defaults. Replace these with the ALL_* lists for the full matrix.
SELECTED_METHODS = ["sparsegpt"]
SELECTED_MODELS = ["blip2"]
SELECTED_COMPRESSION_COMPONENTS = ["V"]

CROSSCODER_COMPONENT = {"V": "V", "P": "P", "V+P": "V_P"}

compression_jobs = [
    (model, method, component)
    for model in SELECTED_MODELS
    for method in SELECTED_METHODS
    for component in SELECTED_COMPRESSION_COMPONENTS
]
print(f"Selected compression jobs: {len(compression_jobs)}")
for job in compression_jobs:
    print("  ", job)

## 3. Create compressed checkpoints

The repository's compression runner normally executes every registered model, method, and component. This cell temporarily filters its in-memory configuration to the selected matrix, then restores it afterward. Checkpoints are written to `src/compressed_models/{model}__{method}__{component}` and completion is recorded in `src/pipeline_log.json`, so reruns skip completed jobs.

In [ ]:
RUN_COMPRESSION = False

if RUN_COMPRESSION:
    import compression_configs
    import compression_utils

    original_methods = compression_configs.METHODS
    original_models = compression_configs.MODEL_CONFIGS
    original_combos = compression_configs.COMPONENT_COMBOS
    try:
        compression_configs.METHODS = list(SELECTED_METHODS)
        compression_configs.MODEL_CONFIGS = {
            name: original_models[name] for name in SELECTED_MODELS
        }
        compression_configs.COMPONENT_COMBOS = {
            label: original_combos[label]
            for label in SELECTED_COMPRESSION_COMPONENTS
        }
        compression_utils.run_compression(quick=False)
    finally:
        compression_configs.METHODS = original_methods
        compression_configs.MODEL_CONFIGS = original_models
        compression_configs.COMPONENT_COMBOS = original_combos
else:
    print("Dry run only. Set RUN_COMPRESSION = True to create checkpoints.")

In [ ]:
compressed_root = SRC_DIR / "compressed_models"
for model, method, component in compression_jobs:
    label = component.replace("+", "_")
    checkpoint = compressed_root / f"{model}__{method}__{label}"
    meta_path = checkpoint / "meta.json"
    status = "ready" if meta_path.exists() else "missing"
    print(f"[{status}] {checkpoint}")
    if meta_path.exists():
        metadata = json.loads(meta_path.read_text(encoding="utf-8"))
        print("  config:", metadata.get("config"))

## 4. Run crosscoder experiments

Each run executes `extract → train → analyze → visualize` and writes to:

`src/crosscoder/results/{model}__{method}__{component}__{token_type}/`

Existing activations, final checkpoints, aggregate metrics, and plots are skipped automatically. Vision experiments use both `cls` and `patch`; projector and combined V+P experiments use `cls`.

In [ ]:
def crosscoder_token_types(component: str) -> list[str]:
    return ["cls", "patch"] if component == "V" else ["cls"]

crosscoder_jobs = []
for model, method, compression_component in compression_jobs:
    component = CROSSCODER_COMPONENT[compression_component]
    for token_type in crosscoder_token_types(component):
        crosscoder_jobs.append((model, method, component, token_type))

print(f"Selected crosscoder jobs: {len(crosscoder_jobs)}")
for job in crosscoder_jobs:
    print("  ", job)

In [ ]:
RUN_CROSSCODER = False
CROSSCODER_STAGE = "all"  # or: extract, train, analyze, visualize

if RUN_CROSSCODER:
    for model, method, component, token_type in crosscoder_jobs:
        command = [
            sys.executable,
            "-m",
            "src.crosscoder.main",
            "--model", model,
            "--method", method,
            "--component", component,
            "--token_type", token_type,
            "--stage", CROSSCODER_STAGE,
        ]
        print("\n$", " ".join(command))
        subprocess.run(command, cwd=REPO_ROOT, check=True)
else:
    print("Dry run only. Set RUN_CROSSCODER = True to execute the jobs above.")

## 5. Inspect tracked results

The cell below reports which stages exist for each selected run. To execute the complete new-method matrix, set:

```python
SELECTED_METHODS = NEW_METHODS
SELECTED_MODELS = ALL_MODELS
SELECTED_COMPRESSION_COMPONENTS = ALL_COMPRESSION_COMPONENTS
```

Then rerun the matrix, compression, and crosscoder cells in order. The full matrix contains 27 compression checkpoints and 36 crosscoder configurations.

In [ ]:
results_root = SRC_DIR / "crosscoder" / "results"
for model, method, component, token_type in crosscoder_jobs:
    run_name = f"{model}__{method}__{component}__{token_type}"
    run_dir = results_root / run_name
    artifacts = {
        "activations": (run_dir / "activations").exists(),
        "trained": (run_dir / "checkpoints" / "final.pt").exists(),
        "analyzed": (run_dir / "metrics" / "aggregate_metrics.json").exists(),
        "visualized": (run_dir / "plots" / "loss_curves.png").exists(),
    }
    print(run_name, artifacts)
    metrics_path = run_dir / "metrics" / "aggregate_metrics.json"
    if metrics_path.exists():
        metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
        print("  aggregate metrics:", metrics)